# 12F-03 — EPA BEV electric-energy EDA

**Question:** Which electric-consumption values are trustworthy, and why?

In [ ]:
from pathlib import Path
import pandas as pd

search_roots = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(path for path in search_roots if (path / "AGENTS.md").exists() and (path / "etl").exists())
SANITY_FILE = REPO_ROOT / "etl/data/processed/sprint_12e3_electric_unit_audit/electric_energy_sanity_dataset.csv"
electric = pd.read_csv(SANITY_FILE, low_memory=False)
electric.shape


## Separate deterministic EPA rows from unresolved and Hydrogen cases

In [ ]:
electric["sprint_12e3a_disposition"].value_counts(dropna=False)


In [ ]:
resolved = electric[
    electric["sprint_12e3a_disposition"].eq("SOURCE_ROW_RESOLVED_KWH_PER_100MI")
    & electric["source_system"].eq("EPA_TESTCAR_2014_PRESENT")
].copy()
resolved["raw_value"] = pd.to_numeric(resolved["raw_value"], errors="raise")
resolved["canonical_wh_per_km"] = pd.to_numeric(resolved["canonical_wh_per_km"], errors="raise")
print(f"{len(resolved)} resolved EPA source records")


## Reproduce the unit conversion visibly

In [ ]:
MI_TO_KM = 1.609344
resolved["notebook_wh_per_km"] = resolved["raw_value"] * 1000 / (100 * MI_TO_KM)
resolved["difference_wh_per_km"] = resolved["notebook_wh_per_km"] - resolved["canonical_wh_per_km"]
resolved[["raw_value", "canonical_wh_per_km", "notebook_wh_per_km", "difference_wh_per_km"]].head()


In [ ]:
assert resolved["difference_wh_per_km"].abs().max() < 1e-9
assert not resolved["electrification"].eq("FCEV").any()
resolved["canonical_wh_per_km"].describe()


## Distribution and extremes

In [ ]:
import matplotlib.pyplot as plt

ax = resolved["canonical_wh_per_km"].plot.hist(bins=25, figsize=(7, 4))
ax.set(title="Resolved EPA electric source observations", xlabel="Wh/km")
plt.show()


In [ ]:
resolved.nlargest(8, "canonical_wh_per_km")[["make", "model", "model_year", "raw_value", "canonical_wh_per_km"]]


## What did we learn?

The 527 exact source-row matches are deterministic at the unit layer. FCEV zeros and non-matching/mixed EPA contexts remain separate; unit closure alone does not create FuelCons.